# Phase 2 — Learned detector (Colab, GPU runtime)

Trains the frozen-DINOv2 + attention-pooling detector and wires it into the fusion
pipeline. All logic lives in repo modules (`training/`, `ai_image_id/detector/`);
this notebook only orchestrates. **Runtime → Change runtime type → GPU (T4).**

Storage policy: immutable sources (GenImage zips) + small versioned artifacts
(checkpoints, calibration) in Drive; large derived caches (extracted images,
prepared splits, embedding shards) on session disk — rebuildable from the zips.
Absolute paths everywhere (`/content/...`) — cells must not depend on the cwd.

In [ ]:
# ── Setup — fresh-kernel-proof ──────────────────────────────────
!apt-get -qq install -y libimage-exiftool-perl p7zip-full > /dev/null
!git clone -q https://github.com/Waranika/AI-image-Checkers.git 2>/dev/null || echo "already cloned"
%cd /content/AI-image-Checkers
%pip install -q -e .
import sys; sys.path.insert(0, "/content/AI-image-Checkers")   # exposes training/

# Verify GPU torch (Colab ships it preinstalled+matched — NEVER pip install torch here)
import torch
print(torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "→ Runtime > Change runtime type > GPU")

## (Optional) Smoke test — full torch chain on synthetic data, ~2 min

No downloads, no Drive. Run this on any fresh runtime before burning GPU-hours:
it exercises `prepare_split → precompute → train → calibrate` end-to-end on 60
synthetic images/class. Numbers are meaningless; execution is everything.

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image

rng = np.random.default_rng(0)
for cls in ["real", "fake"]:
    d = Path(f"/content/smoke/src_{cls}"); d.mkdir(parents=True, exist_ok=True)
    for i in range(60):
        y, x = np.mgrid[0:384, 0:384]
        f1, f2 = rng.uniform(40, 140, 2)
        img = np.stack([120+60*np.sin(x/f1), 100+50*np.cos(y/f2), 90+40*np.sin((x+y)/97)], axis=-1)
        noise = rng.normal(0, 6 if cls == "real" else 2, img.shape)  # learnable class difference
        Image.fromarray(np.clip(img+noise, 0, 255).astype(np.uint8)).save(d/f"{i}.png")

from training.prepare_data import prepare_split
from training.embed import precompute
from training.train_head import train
from training.calibrate_eval import calibrate

prepare_split("/content/smoke/src_real", "/content/smoke/src_fake", "/content/smoke/train", n_per_class=48, seed=0)
prepare_split("/content/smoke/src_real", "/content/smoke/src_fake", "/content/smoke/val",   n_per_class=12, seed=1)
precompute("/content/smoke/train/manifest.csv", "/content/smoke/train", "/content/smoke/emb_train", n_aug=1, device="cuda")
precompute("/content/smoke/val/manifest.csv",   "/content/smoke/val",   "/content/smoke/emb_val",   n_aug=0, device="cuda")
ckpt = train("/content/smoke/emb_train", "/content/smoke/emb_val", out="/content/smoke/head.pt", epochs=3)
print(calibrate("/content/smoke/head.val_logits.npz", "/content/smoke/head.calibration.json"))

## Step 1 — Data: GenImage SDv1.4, val-split slice

The full subset is a **30-volume split zip (~90 GB)** — exceeds session disk.
First-run strategy: extract only `val/*` (6K/class, ~3.6 GB) directly off the
Drive mount with 7z include patterns; the full `train/` run needs a
materialization route (HF mirror / bigger disk) and comes later.

One-time prerequisite in the Drive web UI: open the official GenImage folder
(link in https://github.com/GenImage-Dataset/GenImage), right-click
`stable_diffusion_v_1_4` → **Organiser → Ajouter un raccourci → Mon Drive**.
Shortcuts cost zero quota; shared-with-me files are NOT visible in Colab without one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SHARED = "/content/drive/MyDrive/stable_diffusion_v_1_4"

# Clean slate + sanity checks (fail here, not 30 minutes from now)
!rm -rf /content/zips /content/genimage /content/data
!df -h /content | tail -1                      # want ~60 GB available
!ls "$SHARED"/imagenet_ai_0419_sdv4.* | wc -l  # expect 30 (z01..z29 + .zip)

# Peek inside (reads the central directory only; entry rows may take a minute over FUSE)
!7z l "$SHARED/imagenet_ai_0419_sdv4.zip" | grep -E "Multivolume|Volumes|Total Physical"

# Extract ONLY val, straight off the mount — no copy. 7z chains into whichever
# volumes hold val bytes. ~3.6 GB out; minutes if val sits in the tail volumes.
!7z x -y -o/content/genimage "$SHARED/imagenet_ai_0419_sdv4.zip" \
      "imagenet_ai_0419_sdv4/val/*" | tail -4

# Verify layout BEFORE preparing — trust this output over any README diagram
!find /content/genimage -maxdepth 4 -type d
from pathlib import Path
root = Path("/content/genimage/imagenet_ai_0419_sdv4")   # adjust if find disagrees

## Step 1b — De-confounded, disjoint splits

Single prep of the whole val pool, then **disjoint slices** of the shuffled
manifest — overlap is structurally impossible (two independent seeded draws
overlapped by ~1.7K files when we tried; measured, then fixed). The leakage
guard stays as a permanent tripwire.

In [ ]:
from training.prepare_data import prepare_split
import csv, shutil
from pathlib import Path
from collections import defaultdict

# One prep of the full val pool (6000/class), single seed → files come out shuffled
prepare_split(root/"val/nature", root/"val/ai", "/content/data/all",
              n_per_class=6_000, seed=0)

# Disjoint split: first 5000/class → train, last 1000/class → val
src = Path("/content/data/all")
rows = list(csv.DictReader((src/"manifest.csv").open()))
by_label = defaultdict(list)
for r in rows:
    by_label[r["label"]].append(r)

for split, sl in (("train", slice(0, 5000)), ("val", slice(5000, 6000))):
    out = Path(f"/content/data/{split}")
    out_rows = []
    for label, rs in by_label.items():
        (out/label).mkdir(parents=True, exist_ok=True)
        for r in rs[sl]:
            shutil.move(str(src/label/r["file"]), out/label/r["file"])
            out_rows.append(r)
    with (out/"manifest.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file","label","src","jpeg_q","short_side"])
        w.writeheader(); w.writerows(out_rows)
shutil.rmtree(src)

# Leakage guard — must print 0
train_manifest = "/content/data/train/manifest.csv"
val_manifest   = "/content/data/val/manifest.csv"
train_srcs = {r["src"] for r in csv.DictReader(open(train_manifest))}
val_srcs   = {r["src"] for r in csv.DictReader(open(val_manifest))}
overlap = len(train_srcs & val_srcs)
print(f"train/val source overlap: {overlap} files", "⚠️ LEAKAGE" if overlap else "✓ clean")

## Step 2 — Precompute frozen embeddings (GPU, one pass)

Val first (2K images, n_aug=0, minutes) — it doubles as the real-data smoke test.
Then train (10K × 3 variants = 30K forward passes — the long step). Shards land
on session disk (~6 GB at this scale); they're a rebuildable cache.

In [ ]:
from training.embed import precompute

precompute("/content/data/val/manifest.csv",   "/content/data/val",   "/content/emb/val",
           n_aug=0, device="cuda")
precompute("/content/data/train/manifest.csv", "/content/data/train", "/content/emb/train",
           n_aug=2, device="cuda")

## Step 3 — Train the head (minutes) + calibrate

In [ ]:
from training.train_head import train
from training.calibrate_eval import calibrate

ckpt = train("/content/emb/train", "/content/emb/val", out="/content/head.pt",
             epochs=5, device="cuda")
report = calibrate("/content/head.val_logits.npz", "/content/head.calibration.json")
print(report)   # temperature, ECE before/after
# NOTE: this first run trains and validates within the GenImage val split —
# fine for pipeline validation; publishable numbers need train/ + val/ disjoint
# by the dataset's own construction.

## Step 4 — Persist artifacts to Drive, versioned by commit

Checkpoints are the artifact whose loss hurts; they're small. Version by commit
hash so every result is traceable to the exact code that produced it.

In [ ]:
import json, shutil, subprocess
from pathlib import Path

commit = subprocess.run(["git", "-C", "/content/AI-image-Checkers", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
RUN = Path(f"/content/drive/MyDrive/ai_image_id/runs/{commit}")
RUN.mkdir(parents=True, exist_ok=True)

for f in ["/content/head.pt", "/content/head.calibration.json",
          "/content/data/train/manifest.csv", "/content/data/val/manifest.csv"]:
    shutil.copy(f, RUN)

(RUN/"provenance.json").write_text(json.dumps({
    "commit": commit, "backbone": "dinov2_vits14",
    "data": "GenImage sdv14 val-split slice", "n_train_per_class": 5000,
    "n_val_per_class": 1000, "n_aug": 2, "seeds": {"prep": 0},
}, indent=2))
print("saved to", RUN)

## Step 5 — Use the detector in the pipeline

The fusion engine caps classifier evidence at `likely` (never `verified`),
down-weights on recompression drift, and a confidently-low score unlocks
`unlikely`.

In [ ]:
from ai_image_id.main import analyze_image

r = analyze_image("/content/data/val/fake/" + next(iter(__import__('os').listdir("/content/data/val/fake"))),
                  detector_ckpt="/content/head.pt")
print(r.ai_verdict.value, r.confidence)
print(r.evidence.detector)
print(r.notes)

## Next sessions
1. **Cross-generator eval**: extract other generators' `val/*` slices (Midjourney,
   ADM, BigGAN, GLIDE — same 7z pattern trick), embed, and build the table with
   `training.calibrate_eval.cross_generator_table`. Baseline to beat: GenImage's
   ResNet-50 gets ~52–55% on unseen generators.
2. **Real train/ run**: materialize a random 20K/class from `train/` (HF mirror
   with per-file access, or one-time extraction on a big disk → re-upload the
   sample to Drive), train on it, validate on `val/` — disjoint by construction.
3. **ONNX export** + pin the detector into CI with a tiny fixture.